In [1]:
import os
os.environ["OPENAI_API_KEY"] = 'DUMMY'
os.environ["OLLAMA_HOST"] = 'http://192.168.1.42:11434/v1' # or http://localhost:11434/v1
os.environ["OLLAMA_TIMEOUT"] = '1200'

In [2]:
import json
import pickle
from pageindex import *
# from pageindex.page_index_md import md_to_tree
from pageindex.utils import ConfigLoader, llm_acompletion, create_node_mapping
from pprint import pprint

In [6]:
import nest_asyncio
nest_asyncio.apply()
import asyncio

### Config

In [7]:
user_opt = {
    'model':"openai/gpt-oss:20b", # gpt-oss:20b",
    'retrieve_model':"openai/gpt-oss:20b",
    'if_add_node_id':"yes",
    'if_add_node_summary':"yes",
    'if_add_doc_description':"no",
    'if_add_node_text':"yes",
}

opt = ConfigLoader().load({k: v for k, v in user_opt.items() if v is not None})

## Step 1: PageIndex Tree Generation

In [8]:
%%time
md_path = './data\\2077.md'
toc_with_page_number = asyncio.run(md_to_tree(
    md_path=md_path,
    if_thinning=False,
    min_token_threshold=5000,
    if_add_node_summary=opt.if_add_node_summary,
    summary_token_threshold=200,
    model=opt.model,
    if_add_doc_description=opt.if_add_doc_description,
    if_add_node_text=opt.if_add_node_text,
    if_add_node_id=opt.if_add_node_id
))

Extracting nodes from markdown...
Extracting text content from nodes...
Building tree from nodes...
Formatting tree structure...
Generating summaries for each node...
CPU times: total: 562 ms
Wall time: 50.3 s


In [9]:
toc_with_page_number

{'doc_name': '2077',
 'line_count': 181,
 'structure': [{'title': 'A Cyberpunk 2077 perspective on the prediction and understanding of future technology☆',
   'node_id': '0000',
   'line_num': 1,
   'text': '# A Cyberpunk 2077 perspective on the prediction and understanding of future technology☆\n\nMiguel Bordallo López, Constantino Álvarez Casado\nUniversity of Oulu, Oulu, Finland',
   'nodes': [{'title': 'Highlights',
     'node_id': '0001',
     'line_num': 6,
     'text': '## Highlights\n\nCyberpunk 2077 offers insights into AI, edge computing, biotech, and augmented humans.\nKey developments include BCIs, multimodal systems, VR, and AI-driven appliances.\nThere is a growing need for technologies compatible with existing infrastructure.\nSci-fi games like Cyberpunk 2077 help shape public views on future innovations.',
     'summary': '## Highlights\n\nCyberpunk 2077 offers insights into AI, edge computing, biotech, and augmented humans.\nKey developments include BCIs, multimodal sy

#### 1.1 Save or restore TOC with page number

In [10]:
with open("data/2077_result_w_text_md.pkl", 'wb') as f:
    pickle.dump(toc_with_page_number, f)

In [9]:
# with open("data/2077_result_w_text_md.pkl", 'rb') as f:
#     toc_with_page_number = pickle.load(f)

#### 1.2 Get the generated PageIndex tree structure

In [11]:
utils.list_to_tree(toc_with_page_number['structure'])

[{'title': 'A Cyberpunk 2077 perspective on the prediction and understanding of future technology☆',
  'start_index': None,
  'end_index': None}]

In [12]:
# Remove document text chunks from Tree
tree_with_text = toc_with_page_number['structure'].copy()
tree_without_text = utils.remove_fields(tree_with_text, fields=['text'])
# pprint(json.dumps(tree_without_text, indent=2))

In [13]:
# check JSON
utils.print_json(tree_without_text, max_len=50, indent=2)

[
  {
    "title": "A Cyberpunk 2077 perspective on the prediction and...",
    "node_id": "0000",
    "line_num": 1,
    "nodes": [
      {
        "title": "Highlights",
        "node_id": "0001",
        "line_num": 6,
        "summary": "## Highlights\n\nCyberpunk 2077 offers insights into..."
      },
      {
        "title": "Abstract",
        "node_id": "0002",
        "line_num": 13,
        "summary": "The abstract outlines a position paper that examin..."
      },
      {
        "title": "1. Introduction",
        "node_id": "0003",
        "line_num": 23,
        "summary": "The introduction surveys how science‑fiction liter..."
      },
      {
        "title": "2. Literature review",
        "node_id": "0004",
        "line_num": 35,
        "nodes": [
          {
            "title": "2.1. Intersection of science fiction and technolog...",
            "node_id": "0005",
            "line_num": 38,
            "summary": "The section reviews how science fiction informs t

## Step 2: Reasoning-Based Retrieval with Tree Search

#### 2.1 Use LLM for tree search and identify nodes that might contain relevant context

In [14]:
# query = "What is about simulated reality in this game?"
query = "What model was used in preparing this article and why?"

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

In [15]:
tree_search_result = await llm_acompletion(model="openai/gpt-oss:20b", prompt=search_prompt)

#### 2.2 Print retrieved nodes and reasoning process

In [16]:
print(tree_search_result)

{
    "thinking": "The question asks which model was used to prepare the article and why. The only place in the document that mentions a model is in the Conclusion node (node_id 0023), where it states that GPT-4 was used for drafting and formatting. No other node references a model or explains its use. Therefore, node 0023 is the relevant node.",
    "node_list": ["0023"]
}


In [17]:
tree_search_result_json = json.loads(tree_search_result)
# tree_search_result_json

In [18]:
node_map = utils.create_node_mapping(tree_with_text)
# pprint(node_map)

In [19]:
print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}")


Retrieved Nodes:
Node ID: 0023


## Step 3: Answer Generation

#### 3.1 Extract relevant context from retrieved nodes

In [20]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

## 6. Conclusion
In this paper, we have explored the themes and technologies presented in the video game Cyberpunk
2077 as a lens to envision potential future technological advancements. Our thematic analysis,
grounded in the juxtaposition presented in Table 1, and developed in the following sections,
illustrated the value of science fiction and video games as tools for stimulating imagination and
fostering critical discussion about the direction and implications of emerging technologies.
While the games portrayal of the future is undoubtedly speculative and stylized, it offers valuable
insights into the opportunities and challenges associated with various technological advancements.
By examining these themes and technologies, we can better anticipate the potential impact of new
innovations on society, the economy, and culture and inform the development of technologies that are
more inclusive, ethical, and adaptable to a diverse range of needs and preferences.
Ultim

#### 3.2 Generate answer based on retrieved context

In [21]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

answer_prompt_ext = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a detailed answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await llm_acompletion(model="openai/gpt-oss:20b", prompt=answer_prompt_ext)
utils.print_wrapped(answer)

Generated Answer:

**Model used:**
The authors explicitly state that they employed **OpenAI’s GPT‑4** during the preparation of the
manuscript.

**Reason for using GPT‑4:**
According to the “Declaration of Generative AI and AI‑assisted technologies in the writing process”
section, GPT‑4 was used for two main purposes:

1. **Enhancing readability** – The model helped refine the prose, making the article clearer and
more accessible to readers.
2. **Generating LaTeX code** – GPT‑4 produced the necessary LaTeX markup for references, tables, and
subsections, streamlining the formatting process.

After these tasks, the authors reviewed and edited the content themselves, taking full
responsibility for the final publication.
